# Building a LangChain Chain to Summarize Text

In [15]:
import os
import json
from dotenv import load_dotenv
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient
load_dotenv()

tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [12]:
from typing import Dict, Any

@tool(description="Search the web for information")
def web_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query)

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("ollama:nemotron3:33b", temperature=0.15)

agent = create_agent(
    model=model,
    tools=[web_search],
    checkpointer=InMemorySaver(),
)


In [14]:
from langchain.messages import HumanMessage

question = HumanMessage(content="""given the information about  Elon Reeve Musk, I want you to create:
    1. A short summary
    2. two interesting facts about them""")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [16]:
last_message = response['messages'][-1]
print(last_message.content)
print(json.dumps(last_message.usage_metadata, indent=4))
print(json.dumps(last_message.response_metadata, indent=4))

**Summary:**  
Elon Reeve Musk (born June 28, 1971) is a South African-born American entrepreneur and business magnate. He is the CEO and largest shareholder of Tesla and SpaceX, and also founded xAI, The Boring Company, Neuralink, and OpenAI's early partners. Musk became the world’s richest person in 2025, amassing a net worth of $744 billion by mid-2026, making him the first trillionaire in U.S. dollars. He also served briefly as a Senior Advisor to the President for Government Efficiency under Donald Trump in early 2025.

**Two Interesting Facts:**  
1. **Dual Citizenship at Birth**: Musk holds citizenship in South Africa (birthplace) and Canada (through his mother) from birth, later acquiring U.S. citizenship in 2002.  
2. **Trillionaire Milestone**: In June 2026, Musk became the first person in history to reach a net worth of $1 trillion, driven by Tesla and SpaceX valuations, solidifying his status as the world’s wealthiest individual.
{
    "input_tokens": 2540,
    "output_toke